# Raidium Challenge — Segmentation CT abdominale


## 1. Setup

In [ ]:
import torch

# Quick environment check. This notebook is meant to run locally and falls back to CPU if no CUDA device is visible.
if torch.cuda.is_available():
    print('CUDA device :', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM  : {total:.1f} GB')
else:
    print('No CUDA device found - running on CPU')

In [ ]:
# Third-party deps used below. PyTorch is assumed pre-installed with a matching
# CUDA build
!pip install segmentation_models_pytorch albumentations -q

In [ ]:
import os, json, time, random, shutil, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
import albumentations as A
import cv2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DEVICE = DEVICE.type
print('Device :', DEVICE)

def empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Local layout. Place the provided artefacts next to this notebook:
#   ./annotated_labels.json      partial per-image class labels
#   ./cache/                     masks.npy, train_images.npy, test_images.npy
#   ./start_best_model.pt        initialisation weights (not provided)
DATA_DIR   = Path('.').resolve()
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = ['annotated_labels.json', 'cache', 'start_best_model.pt']
missing  = [r for r in required if not (DATA_DIR / r).exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs in {DATA_DIR}: {missing}')
print('All inputs found in', DATA_DIR)

## 2. Config

In [ ]:
NUM_CLASSES = 55
IMG_SIZE = 256
NUM_EPOCHS = 35
LR = 3e-4
WEIGHT_DECAY = 1e-4
# Lots of issues with that one, notebook DataLoaders with workers > 0 can deadlock. set to 0 there.
NUM_WORKERS = 2

LABELED_BS = 8
UNLABELED_BS = 8
BATCH_SIZE = LABELED_BS + UNLABELED_BS

# Dataset-level normalisation stats (precomputed over the training images).
DATASET_MEAN = 0.240
DATASET_STD = 0.194

ALPHA_CE = 0.5

EMA_ALPHA = 0.99
CONSISTENCY_MAX = 1.0
RAMPUP_EPOCHS = 5
CONFIDENCE_THRESH = 0.9
AUGMENT_PROB = 0.5

# Class weights: weight = clip(0.3 / (dice + 0.05), 1, 8), background kept low.
WEIGHT_CLIP_MIN = 1.0
WEIGHT_CLIP_MAX = 8.0
WEIGHT_NUMERATOR = 0.3
WEIGHT_OFFSET = 0.05
WEIGHT_RAMPUP = 5          # linear ramp of class weights over the first epochs
BG_WEIGHT = 0.5           # background (class 0) is always down-weighted

EARLY_STOP_PATIENCE = 12

JSON_PATH       = DATA_DIR / 'annotated_labels.json'
CACHE_DIR       = DATA_DIR / 'cache'
INIT_MODEL_PATH = DATA_DIR / 'start_best_model.pt'   # provided initialisation
BEST_MODEL_PATH = OUTPUT_DIR / 'final_best_model.pt'
SUBMIT_PATH     = OUTPUT_DIR / 'submission.csv'

## 3. Data

In [ ]:
masks = np.load(CACHE_DIR / 'masks.npy')
train_images = np.load(CACHE_DIR / 'train_images.npy')
test_images = np.load(CACHE_DIR / 'test_images.npy')
with open(JSON_PATH) as f: labels_json = json.load(f)

# Labels are partial: an image is "annotated" only if its class list is non-empty.
annotated_idx = np.array([i for i,L in enumerate(labels_json) if len(L)>0])
unannotated_idx = np.array([i for i,L in enumerate(labels_json) if len(L)==0])

# validity[i, c] = True means class c is actually labelled in image i. Background
# (class 0) is always valid so it can be supervised / masked everywhere.
validities = np.zeros((len(labels_json), NUM_CLASSES), dtype=bool)
for i,L in enumerate(labels_json):
    for c in L:
        if 0<=c<NUM_CLASSES: validities[i,c] = True
    validities[i,0] = True

rng = np.random.default_rng(SEED)
perm = rng.permutation(annotated_idx)
n_val = int(0.2 * len(perm))
val_ids = perm[:n_val]
labeled_train_ids = perm[n_val:]
print(f'labeled_train : {len(labeled_train_ids)} | val : {len(val_ids)}')

## 4. Class weights

In [ ]:
def create_unet():
    # ResNet-18 U-Net, single-channel input (CT slice), NUM_CLASSES outputs.
    return smp.Unet(encoder_name='resnet18', encoder_weights=None,
                     in_channels=1, classes=NUM_CLASSES).to(DEVICE)

# Measure per-class Dice of the provided init weights on val, using the masked
# prediction (M1). I weight on M1 because the genuinely missed classes are the
# same under M1 and M4, and M1 is consistent with the masked supervised loss.
init_temp = create_unet()
init_temp.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=DEVICE))
init_temp.eval()

class ValDataset(Dataset):
    def __init__(self, ids):
        self.ids = np.asarray(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = int(self.ids[idx])
        img = (train_images[i].astype(np.float32) / 255.0 - DATASET_MEAN) / DATASET_STD
        return {
            'image':    torch.from_numpy(img).unsqueeze(0).float(),
            'mask':     torch.from_numpy(masks[i].astype(np.int64)),
            'validity': torch.from_numpy(validities[i].astype(np.float32)),
            'image_id': i,
        }

val_loader = DataLoader(ValDataset(val_ids), batch_size=16, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

per_class_dice_init = defaultdict(list)

with torch.no_grad():
    for batch in val_loader:
        img      = batch['image'].to(DEVICE, non_blocking=True)
        mask     = batch['mask'].to(DEVICE, non_blocking=True)
        validity = batch['validity'].to(DEVICE, non_blocking=True)
        b = img.size(0)
        with torch.amp.autocast(AMP_DEVICE):
            logits = init_temp(img)
        # mask out classes not labelled in this image before argmax
        mask4d = validity.view(b, NUM_CLASSES, 1, 1)
        NEG_INF = torch.finfo(logits.dtype).min / 2
        logits_m = logits.masked_fill(mask4d < 0.5, NEG_INF)
        pred = logits_m.argmax(1)
        for k in range(b):
            valid_c = [c for c in torch.where(validity[k] > 0.5)[0].tolist() if c != 0]
            for c in valid_c:
                if (mask[k] == c).sum() == 0: continue
                inter = ((pred[k] == c) & (mask[k] == c)).sum().float()
                denom = (pred[k] == c).sum().float() + (mask[k] == c).sum().float()
                per_class_dice_init[c].append(((2*inter + 1e-6) / (denom + 1e-6)).item())

dice_per_class_init = {c: float(np.mean(v)) if v else 0.0 for c, v in per_class_dice_init.items()}
del init_temp; empty_cache()

class_weights_target = np.full(NUM_CLASSES, 1.0, dtype=np.float32)
class_weights_target[0] = BG_WEIGHT
for c in range(1, NUM_CLASSES):
    d = dice_per_class_init.get(c, 0.0)
    raw_weight = WEIGHT_NUMERATOR / (d + WEIGHT_OFFSET)
    class_weights_target[c] = float(np.clip(raw_weight, WEIGHT_CLIP_MIN, WEIGHT_CLIP_MAX))

rows = []
for c in range(0, NUM_CLASSES):
    rows.append({
        'classe': c,
        'dice_init': round(dice_per_class_init.get(c, 0.0), 4) if c > 0 else None,
        'weight_target': round(float(class_weights_target[c]), 3),
    })
df_w = pd.DataFrame(rows).sort_values('weight_target', ascending=False).reset_index(drop=True)
print('=== Class weights cibles (top 20) ===')
print(df_w.head(20).to_string(index=False))
print(f'\nNb classes au plafond ({WEIGHT_CLIP_MAX:.0f}) : {(class_weights_target == WEIGHT_CLIP_MAX).sum()}')
print(f'Nb classes au plancher ({WEIGHT_CLIP_MIN:.0f}) : {(class_weights_target[1:] == WEIGHT_CLIP_MIN).sum()}')

In [ ]:
# Visualise target weights: red = strongly up-weighted (hard classes), blue = neutral.
fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(NUM_CLASSES)
colors = ['red' if class_weights_target[c] >= 5 else
          'orange' if class_weights_target[c] >= 2 else
          'lightblue' for c in range(NUM_CLASSES)]
ax.bar(x, class_weights_target, color=colors, alpha=0.7)
ax.axhline(WEIGHT_CLIP_MAX, color='red', linestyle=':', label=f'plafond {WEIGHT_CLIP_MAX}')
ax.axhline(1.0, color='gray', linestyle=':', label='neutre')
ax.set_xlabel('Classe'); ax.set_ylabel('Poids cible')
ax.set_title('Class weights cibles (rouge: >=5, orange: 2-5, bleu: <=2)')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Student + teacher models

In [ ]:
student = create_unet()
teacher = create_unet()
# Both networks start from the provided init weights.
student.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=DEVICE))
teacher.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=DEVICE))

# Teacher is an EMA of the student: never optimised directly
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

@torch.no_grad()
def update_teacher_ema(student, teacher, alpha=EMA_ALPHA):
    for p_t, p_s in zip(teacher.parameters(), student.parameters()):
        p_t.data.mul_(alpha).add_(p_s.data, alpha=1.0 - alpha)
    for b_t, b_s in zip(teacher.buffers(), student.buffers()):
        if b_s.dtype.is_floating_point:
            b_t.data.mul_(alpha).add_(b_s.data, alpha=1.0 - alpha)
        else:
            b_t.data.copy_(b_s.data)

n_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'params : {n_train/1e6:.2f}M')

## 6. Augmentations

In [ ]:
# Geometric + photometric augmentations shared by the labeled training set and
# the student branch of the unlabeled path.
strong_aug = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Affine(rotate=(-15,15), scale=(0.9,1.1),
             translate_percent={'x':(-0.05,0.05),'y':(-0.05,0.05)},
             border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0, p=AUGMENT_PROB),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=AUGMENT_PROB),
    A.RandomGamma(gamma_limit=(80,120), p=AUGMENT_PROB),
    A.CoarseDropout(num_holes_range=(1,3), hole_height_range=(8,16),
                    hole_width_range=(8,16), fill=0, fill_mask=0, p=AUGMENT_PROB),
])

## 7. Datasets

In [ ]:
class LabeledDataset(Dataset):
    def __init__(self, ids, transform=None):
        self.ids = np.asarray(ids); self.transform = transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = int(self.ids[idx])
        img, mask = train_images[i], masks[i]
        if self.transform is not None:
            out = self.transform(image=img, mask=mask)
            img, mask = out['image'], out['mask']
        img_f = (img.astype(np.float32) / 255.0 - DATASET_MEAN) / DATASET_STD
        return {
            'image':    torch.from_numpy(img_f).unsqueeze(0).float(),
            'mask':     torch.from_numpy(mask.astype(np.int64)),
            'validity': torch.from_numpy(validities[i].astype(np.float32)),
            'image_id': i,
        }

class UnlabeledDataset(Dataset):
    # Mean-Teacher consistency: teacher and student see the SAME image with the
    # same random flips but the student additionally gets the strong (affine /
    # photometric / dropout) augmentations. I push the student to agree
    # with the teacher which provides a training signal on unlabelled slices.
    def __init__(self, ids):
        self.ids = np.asarray(ids)
        self.strong_only = A.Compose([
            A.Affine(rotate=(-15,15), scale=(0.9,1.1),
                     translate_percent={'x':(-0.05,0.05),'y':(-0.05,0.05)},
                     border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0, p=AUGMENT_PROB),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=AUGMENT_PROB),
            A.RandomGamma(gamma_limit=(80,120), p=AUGMENT_PROB),
            A.CoarseDropout(num_holes_range=(1,3), hole_height_range=(8,16),
                            hole_width_range=(8,16), fill=0, fill_mask=0, p=AUGMENT_PROB),
        ])
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = int(self.ids[idx])
        img = train_images[i].copy()
        # Shared weak augmentation so teacher and student stay aligned.
        if random.random() < 0.5: img = img[:, ::-1].copy()
        if random.random() < 0.5: img = img[::-1, :].copy()
        teacher_img = img.copy()
        student_img = self.strong_only(image=img, mask=np.zeros_like(img))['image']
        def norm(x):
            x = (x.astype(np.float32) / 255.0 - DATASET_MEAN) / DATASET_STD
            return torch.from_numpy(x).unsqueeze(0).float()
        return {'student_image': norm(student_img),
                'teacher_image': norm(teacher_img),
                'image_id': i}

val_ds       = LabeledDataset(val_ids, transform=None)
labeled_ds   = LabeledDataset(labeled_train_ids, transform=strong_aug)
unlabeled_ds = UnlabeledDataset(unannotated_idx)

labeled_loader   = DataLoader(labeled_ds,   batch_size=LABELED_BS,   shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
unlabeled_loader = DataLoader(unlabeled_ds, batch_size=UNLABELED_BS, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader       = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

## 8. Losses

In [ ]:
def masked_ce_loss(logits, target, validity, weights=None):
    # Classes not labelled in an image are pushed to -inf before the softmax so
    # they cannot win the cross-entropy on that image.
    B, C, H, W = logits.shape
    NEG_INF = torch.finfo(logits.dtype).min / 2
    mask4d = validity.view(B, C, 1, 1)
    logits_m = logits.masked_fill(mask4d < 0.5, NEG_INF)
    if weights is not None: weights = weights.to(logits.dtype)
    return F.cross_entropy(logits_m, target, weight=weights, reduction='mean')

def masked_dice_loss(logits, target, validity, eps=1e-6):
    B, C, H, W = logits.shape
    NEG_INF = torch.finfo(logits.dtype).min / 2
    mask4d = validity.view(B, C, 1, 1)
    logits_m = logits.masked_fill(mask4d < 0.5, NEG_INF)
    probs = F.softmax(logits_m, dim=1)
    target_oh = F.one_hot(target, num_classes=C).permute(0, 3, 1, 2).to(probs.dtype)
    inter = (probs * target_oh).sum(dim=(2, 3))
    denom = probs.sum(dim=(2, 3)) + target_oh.sum(dim=(2, 3))
    dice  = (2 * inter + eps) / (denom + eps)
    # Average Dice only over valid, non-background classes that are present in
    # the ground truth// then average over images that have at least one such class
    valid_no_bg = validity.clone(); valid_no_bg[:, 0] = 0
    present_in_gt = (target_oh.sum(dim=(2, 3)) > 0).to(dice.dtype)
    final_mask = valid_no_bg * present_in_gt
    dice_masked = dice * final_mask
    counts = final_mask.sum(dim=1).clamp(min=1.0)
    has_any = (final_mask.sum(dim=1) > 0).to(dice.dtype)
    dice_per_img = (dice_masked.sum(dim=1) / counts) * has_any
    n_valid_imgs = has_any.sum().clamp(min=1.0)
    return 1.0 - (dice_per_img.sum() / n_valid_imgs)

def supervised_loss(logits, target, validity, weights=None, alpha=ALPHA_CE):
    ce   = masked_ce_loss(logits, target, validity, weights)
    dice = masked_dice_loss(logits, target, validity)
    return alpha * ce + (1 - alpha) * dice, ce.detach().item(), dice.detach().item()

def consistency_loss(student_logits, teacher_logits, threshold=CONFIDENCE_THRESH):
    # Only enforce agreement where the teacher is confident and predicts a
    # foreground class to avoid reinforcing background / uncertain pixels.
    teacher_probs = F.softmax(teacher_logits.detach(), dim=1)
    student_probs = F.softmax(student_logits, dim=1)
    teacher_conf, teacher_argmax = teacher_probs.max(dim=1)
    confident_mask = (teacher_conf >= threshold) & (teacher_argmax != 0)
    confident_mask4d = confident_mask.unsqueeze(1).to(student_probs.dtype)
    sq_diff = (student_probs - teacher_probs) ** 2
    sq_diff_masked = sq_diff * confident_mask4d
    n_confident = confident_mask.sum().clamp(min=1.0)
    loss = sq_diff_masked.sum() / (NUM_CLASSES * n_confident)
    return loss, float(confident_mask.float().mean().item())

def consistency_weight(epoch, max_w=CONSISTENCY_MAX, rampup=RAMPUP_EPOCHS):
    # Gaussian ramp-up so the consistency term does not dominate early training.
    if epoch >= rampup: return max_w
    phase = 1.0 - epoch / rampup
    return float(max_w * np.exp(-5.0 * phase * phase))

def class_weights_at_epoch(epoch, target_weights, rampup=WEIGHT_RAMPUP):
    # Linear ramp of class weights from uniform (1.0) towards target_weights so
    # the strong up-weighting of hard classes is introduced progressively.
    if epoch >= rampup:
        return target_weights.copy()
    alpha = epoch / rampup
    uniform = np.ones_like(target_weights)
    uniform[0] = BG_WEIGHT
    return alpha * target_weights + (1 - alpha) * uniform

## 9. M1 & M4-unmasked

In [ ]:
@torch.no_grad()
def evaluate_M1_and_M4(model_, loader, num_classes=NUM_CLASSES, eps=1e-6):
    """Compute M1 (our usual per-image, validity-masked Dice) and M4-unmasked
    (global macro-Dice with no validity masking). Returns (M1, M4_unmasked).
    M4-unmasked is the metric the platform actually uses."""
    model_.eval()
    # M1: averaged per image.
    m1_dices = []
    # M4: global per-class intersections / denominators accumulated over the set.
    m4_inter = np.zeros(num_classes, dtype=np.float64)
    m4_denom = np.zeros(num_classes, dtype=np.float64)

    for batch in loader:
        img      = batch['image'].to(DEVICE, non_blocking=True)
        mask     = batch['mask'].to(DEVICE, non_blocking=True)
        validity = batch['validity'].to(DEVICE, non_blocking=True)
        b = img.size(0)
        with torch.amp.autocast(AMP_DEVICE):
            logits = model_(img)

        #M1: masked prediction, per image
        mask4d = validity.view(b, num_classes, 1, 1)
        NEG_INF = torch.finfo(logits.dtype).min / 2
        logits_m = logits.masked_fill(mask4d < 0.5, NEG_INF)
        pred_masked = logits_m.argmax(1)
        for k in range(b):
            valid_c = [c for c in torch.where(validity[k] > 0.5)[0].tolist() if c != 0]
            per_img = []
            for c in valid_c:
                if (mask[k] == c).sum() == 0: continue
                inter = ((pred_masked[k] == c) & (mask[k] == c)).sum().float()
                denom = (pred_masked[k] == c).sum().float() + (mask[k] == c).sum().float()
                per_img.append(((2*inter + eps) / (denom + eps)).item())
            if per_img: m1_dices.append(np.mean(per_img))

        # M4: unmasked prediction -> global accumulation
        pred_unmasked = logits.argmax(1)
        for k in range(b):
            for c in range(1, num_classes):
                inter_kc = ((pred_unmasked[k] == c) & (mask[k] == c)).sum().item()
                denom_kc = ((pred_unmasked[k] == c).sum() + (mask[k] == c).sum()).item()
                m4_inter[c] += inter_kc
                m4_denom[c] += denom_kc

    m1_score = float(np.mean(m1_dices)) if m1_dices else 0.0

    # M4: per-class Dice then macro-average, and absent classes count as Dice 0.
    m4_per_class = []
    for c in range(1, num_classes):
        if m4_denom[c] > 0:
            m4_per_class.append((2*m4_inter[c] + eps) / (m4_denom[c] + eps))
        else:
            m4_per_class.append(0.0)
    m4_score = float(np.mean(m4_per_class))

    return m1_score, m4_score

# Baseline: measure the provided init weights on val.
init_temp = create_unet()
init_temp.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=DEVICE))
m1_init, m4_init = evaluate_M1_and_M4(init_temp, val_loader)
del init_temp; empty_cache()
print(f'\n=== Baseline (provided init) sur val ===')
print(f'  M1 (masked, par image)  : {m1_init:.4f}')
print(f'  M4 (unmasked, global)   : {m4_init:.4f}  <- metrique plateforme')
# For reference, the provided init evaluated with TTA scored 0.4005 on the platform.
print(f'  Reference plateforme (init + TTA) : 0.4005')

## 10. Training

In [ ]:
trainable_params = [p for p in student.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler(AMP_DEVICE)

history = {'sup':[], 'cons':[], 'lambda_c':[], 'weight_alpha':[],
           'val_M1':[], 'val_M4':[]}
best_m4 = 0.0
best_epoch = 0
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    student.train()
    teacher.eval()

    # Class weights for this epoch
    cw = class_weights_at_epoch(epoch - 1, class_weights_target)
    cw_tensor = torch.from_numpy(cw).to(DEVICE)
    weight_alpha = (epoch - 1) / WEIGHT_RAMPUP if epoch <= WEIGHT_RAMPUP else 1.0

    lam_cons = consistency_weight(epoch - 1)
    sums = {'sup':0.0, 'cons':0.0, 'n_steps':0}

    # The unlabeled loader is iterated alongside the labeled one and I restart it
    # whenever it is exhausted so every labeled batch is paired with one.
    unlabeled_iter = iter(unlabeled_loader)
    for lb in labeled_loader:
        try:
            ub = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            ub = next(unlabeled_iter)

        img_lab   = lb['image'].to(DEVICE, non_blocking=True)
        mask_lab  = lb['mask'].to(DEVICE, non_blocking=True)
        valid_lab = lb['validity'].to(DEVICE, non_blocking=True)
        img_stud_unlab = ub['student_image'].to(DEVICE, non_blocking=True)
        img_teach_unlab= ub['teacher_image'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(AMP_DEVICE):
            logits_lab = student(img_lab)
            sup_loss, _, _ = supervised_loss(logits_lab, mask_lab, valid_lab, cw_tensor)
            logits_unlab_s = student(img_stud_unlab)

        # Teacher forward stays outside the graph (no grad, detached target).
        with torch.no_grad():
            with torch.amp.autocast(AMP_DEVICE):
                logits_unlab_t = teacher(img_teach_unlab)

        with torch.amp.autocast(AMP_DEVICE):
            cons_loss, _ = consistency_loss(logits_unlab_s, logits_unlab_t)
            total_loss = sup_loss + lam_cons * cons_loss

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
        scaler.step(optimizer); scaler.update()

        # Update the teacher after the student step (EMA of the new weights).
        update_teacher_ema(student, teacher, alpha=EMA_ALPHA)

        sums['sup']  += sup_loss.item()
        sums['cons'] += cons_loss.item()
        sums['n_steps'] += 1

    avg = {k: sums[k]/sums['n_steps'] for k in ['sup','cons']}

    # Evaluate both student and teacher on both metrics.
    m1_s, m4_s = evaluate_M1_and_M4(student, val_loader)
    m1_t, m4_t = evaluate_M1_and_M4(teacher, val_loader)

    # Model selection is driven by M4 (the true platform metric), taking the
    # better of student / teacher at each epoch.
    best_of_two_m4 = max(m4_s, m4_t)

    scheduler.step()
    history['sup'].append(avg['sup']); history['cons'].append(avg['cons'])
    history['lambda_c'].append(lam_cons); history['weight_alpha'].append(weight_alpha)
    history['val_M1'].append(max(m1_s, m1_t))
    history['val_M4'].append(best_of_two_m4)

    flag = ''
    if best_of_two_m4 > best_m4:
        best_m4 = best_of_two_m4
        best_epoch = epoch
        which = 'teacher' if m4_t >= m4_s else 'student'
        target_model = teacher if which == 'teacher' else student
        torch.save(target_model.state_dict(), BEST_MODEL_PATH)
        flag = f' *SAVED ({which})*'
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    dt = time.time() - t0
    print(f'E{epoch:02d}/{NUM_EPOCHS} | sup {avg["sup"]:.3f} cons {avg["cons"]:.4f} '
          f'lc {lam_cons:.2f} a {weight_alpha:.2f} | '
          f'M1_S {m1_s:.4f} M1_T {m1_t:.4f} | M4_S {m4_s:.4f} M4_T {m4_t:.4f} | '
          f'{dt:5.1f}s{flag}')

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping (best E{best_epoch})')
        break

print(f'\nBest M4 = {best_m4:.4f} (epoch {best_epoch})')
print(f'   Reference init : M4 = {m4_init:.4f}')
print(f'   Delta : {best_m4 - m4_init:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
ep = np.arange(1, len(history['sup'])+1)
axes[0].plot(ep, history['sup'],  label='sup loss')
axes[0].plot(ep, history['cons'], label='cons loss')
axes[0].set_title('Losses'); axes[0].legend(fontsize=8)
axes[1].plot(ep, history['lambda_c'], label='lambda cons')
axes[1].plot(ep, history['weight_alpha'], label='alpha weights')
axes[1].set_title('Rampes'); axes[1].legend(fontsize=8)
axes[2].plot(ep, history['val_M1'], label='M1')
axes[2].plot(ep, history['val_M4'], label='M4')
axes[2].axhline(m4_init, color='red', linestyle=':', label=f'init baseline M4: {m4_init:.3f}')
axes[2].axhline(0.4005, color='orange', linestyle=':', label='init+TTA: 0.4005')
axes[2].set_title('Val M1 vs M4'); axes[2].legend(fontsize=8)
for a in axes: a.set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 11. Output

In [ ]:
@torch.no_grad()
def predict_softmax_tta(model, image_batch):
    # 4-way flip TTA (identity, H, V, HV), averaged in probability space.
    sm_orig = F.softmax(model(image_batch).float(), dim=1)
    img_h = torch.flip(image_batch, dims=[-1])
    sm_h = F.softmax(model(img_h).float(), dim=1)
    sm_h = torch.flip(sm_h, dims=[-1])
    img_v = torch.flip(image_batch, dims=[-2])
    sm_v = F.softmax(model(img_v).float(), dim=1)
    sm_v = torch.flip(sm_v, dims=[-2])
    img_hv = torch.flip(image_batch, dims=[-1, -2])
    sm_hv = F.softmax(model(img_hv).float(), dim=1)
    sm_hv = torch.flip(sm_hv, dims=[-1, -2])
    return (sm_orig + sm_h + sm_v + sm_hv) / 4.0

@torch.no_grad()
def evaluate_M4_with_tta(model_, loader, num_classes=NUM_CLASSES, eps=1e-6):
    model_.eval()
    m4_inter = np.zeros(num_classes, dtype=np.float64)
    m4_denom = np.zeros(num_classes, dtype=np.float64)
    for batch in loader:
        img = batch['image'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(AMP_DEVICE):
            sm = predict_softmax_tta(model_, img)
        pred = sm.argmax(1)
        for k in range(img.size(0)):
            for c in range(1, num_classes):
                inter_kc = ((pred[k] == c) & (mask[k] == c)).sum().item()
                denom_kc = ((pred[k] == c).sum() + (mask[k] == c).sum()).item()
                m4_inter[c] += inter_kc
                m4_denom[c] += denom_kc
    m4_per_class = []
    for c in range(1, num_classes):
        if m4_denom[c] > 0:
            m4_per_class.append((2*m4_inter[c] + eps) / (m4_denom[c] + eps))
        else:
            m4_per_class.append(0.0)
    return float(np.mean(m4_per_class))

# Fine-tuned model (this run): raw and with TTA.
best_model = create_unet()
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
_, m4_run = evaluate_M1_and_M4(best_model, val_loader)
m4_run_tta = evaluate_M4_with_tta(best_model, val_loader)

# fallback submission.
init_model = create_unet()
init_model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=DEVICE))
m4_init_tta = evaluate_M4_with_tta(init_model, val_loader)

print(f'  init (reference)   : M4 = {m4_init:.4f}')
print(f'  init + TTA         : M4 = {m4_init_tta:.4f}')
print(f'  run (fine-tuned)   : M4 = {m4_run:.4f}')
print(f'  run + TTA          : M4 = {m4_run_tta:.4f}')

best_score = max(m4_init_tta, m4_run, m4_run_tta)
if best_score == m4_run_tta and m4_run_tta > m4_init_tta:
    print('\n[OK] run + TTA est le meilleur -> soumission run + TTA')
    inference_fn = lambda x: predict_softmax_tta(best_model, x)
    submission_label = 'finetuned_tta'
elif best_score == m4_run and m4_run > m4_init_tta:
    print('\n[OK] run brut est le meilleur -> soumission run')
    inference_fn = lambda x: F.softmax(best_model(x).float(), dim=1)
    submission_label = 'finetuned'
else:
    print('\n[WARN] init + TTA reste le meilleur -> soumission init + TTA (securite)')
    inference_fn = lambda x: predict_softmax_tta(init_model, x)
    submission_label = 'init_tta_safe'

del init_model; empty_cache()

## 12. Inférence test SANS masquage validity

In [ ]:
class TestDataset(Dataset):
    def __init__(self, ids):
        self.ids = np.asarray(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = int(self.ids[idx])
        img = (test_images[i].astype(np.float32) / 255.0 - DATASET_MEAN) / DATASET_STD
        return {'image': torch.from_numpy(img).unsqueeze(0).float(), 'image_id': i}

test_loader = DataLoader(TestDataset(np.arange(len(test_images))),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
predictions = np.zeros((len(test_images), IMG_SIZE, IMG_SIZE), dtype=np.uint8)

# No validity masking on test: none is available, and the platform metric is
# unmasked, so we take a plain argmax over all classes (consistent with val M4).
with torch.no_grad():
    for batch in test_loader:
        img = batch['image'].to(DEVICE, non_blocking=True)
        ids = batch['image_id'].numpy()
        with torch.amp.autocast(AMP_DEVICE):
            sm = inference_fn(img)
        pred = sm.argmax(1).cpu().numpy().astype(np.uint8)
        for k, i in enumerate(ids): predictions[i] = pred[k]

print(f'Predictions test : {predictions.shape}, classes uniques : {np.unique(predictions).size}')

In [ ]:
test_files = [f'{i}.png' for i in range(len(test_images))]

# Official submission format (integer columns).
path_A = str(SUBMIT_PATH).replace('.csv', f'_{submission_label}_officiel.csv')
pd.DataFrame(predictions.reshape((predictions.shape[0], -1))).T.to_csv(path_A)

# Human-readable variant (named pixel rows / file columns), for inspection.
path_B = str(SUBMIT_PATH).replace('.csv', f'_{submission_label}_named.csv')
pd.DataFrame(
    predictions.reshape((predictions.shape[0], -1)).T,
    index   = [f'Pixel {k}' for k in range(IMG_SIZE*IMG_SIZE)],
    columns = test_files,
).to_csv(path_B)

def read_image_from_csv(path, i):
    df = pd.read_csv(path, index_col=0, header=0).T
    try:    return df.iloc[i].values.reshape((IMG_SIZE, IMG_SIZE))
    except: return df.loc[i].values.reshape((IMG_SIZE, IMG_SIZE))

# some issus with submission website : re-read the CSV and confirm a couple of images match exactly.
for name, p in [('officiel', path_A), ('named', path_B)]:
    chk = pd.read_csv(p, index_col=0)
    print(f'[{name}] shape={chk.shape}')
    for i in [0, len(test_files)-1]:
        assert np.array_equal(read_image_from_csv(p, i), predictions[i])
print(f'\nSoumissions generated with label "{submission_label}" dans {OUTPUT_DIR}.')